In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

In [2]:
import pandas as pd
import re
import numpy as np
import spacy

/home/coler/projects/edaic_ftr/.venv/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/coler/projects/edaic_ftr/.venv/lib/python3.9/site-packages/spacy/cli/info.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
df = pd.read_csv("../data/edaic_ALL_participant_features.csv", index_col=0)
df = df.reset_index()
df.columns = [x.lower() for x in df.columns]
df.head()

,participant_id,split,gender,age,language,is_english,en_token_ratio,es_token_ratio,transcript_text,n_utterances,...,ftr_n_temporal,ftr_temporal_rate_1k,ftr_mean_log_horizon,ftr_median_abs_days,ftr_future_frac,ftr_n_future,ftr_n_past,tense_past_rate,tense_pres_rate,tense_n_finite
0,300,dev,male,33,en,True,0.98,0.16,so I'm going to interview in Spanish okay good...,77,...,2.0,6.230530,2.639057,14.0,0.0,0.0,2.0,0.316667,0.683333,60.0
1,301,dev,male,39,en,True,0.93,0.11,yeah there's also on Craigslist so that's why ...,72,...,1.0,0.714286,0.000000,1.0,0.0,0.0,1.0,0.135849,0.864151,265.0
2,302,train,male,25,en,True,0.99,0.10,just move around a little bit when you're fini...,99,...,4.0,6.578947,0.934417,1.5,0.0,0.0,4.0,0.220930,0.779070,86.0
3,303,train,female,41,en,True,0.96,0.09,wow okay when you're finished when she's done ...,94,...,2.0,1.044932,1.945910,7.0,1.0,2.0,0.0,0.182058,0.817942,379.0
4,304,train,female,22,en,True,0.96,0.11,so we'll just move around a little bit tonight...,80,...,5.0,5.055612,4.578530,365.0,0.4,2.0,3.0,0.207865,0.792135,178.0


In [5]:
#flatten sentences
from tqdm import tqdm as prog_bar
from time import time
import spacy

def timer_func(func):
    # This function shows the execution time of 
    # the function object passed
    def wrap_func(*args, **kwargs):
        t1 = time()
        result = func(*args, **kwargs)
        t2 = time()
        print(f'Function {func.__name__!r} executed in {(t2-t1):.4f}s')
        return result
    return wrap_func

# Chunk into sentences

In [49]:
import ast

@timer_func
def sentencize(df, id_col, text_col):

    dx = df.copy()

    records = [
        {
            id_col: idx,
            'utterance_id': utt_id,
            'utterance': utt['text'],
            'start_time': utt.get('start_time'),
            'end_time': utt.get('end_time'),
            'confidence': utt.get('confidence'),
        }
        for idx, raw in zip(dx[id_col], dx[text_col])
        for utt_id, utt in enumerate(ast.literal_eval(raw))
    ]
    sents = pd.DataFrame(records)
    return pd.merge(dx, sents, on=id_col)


In [50]:
sents = sentencize(df,"participant_id", "utterances")

Function 'sentencize' executed in 0.9365s


In [51]:
sents.shape

(26083, 162)

In [52]:
sents.head()

,participant_id,split,gender,age,language,is_english,en_token_ratio,es_token_ratio,transcript_text,n_utterances,...,ftr_n_future,ftr_n_past,tense_past_rate,tense_pres_rate,tense_n_finite,utterance_id,utterance,start_time,end_time,confidence
0,300,dev,male,33,en,True,0.98,0.16,so I'm going to interview in Spanish okay good...,77,...,0.0,2.0,0.316667,0.683333,60.0,0,so I'm going to,14.3,15.1,0.934210
1,300,dev,male,33,en,True,0.98,0.16,so I'm going to interview in Spanish okay good...,77,...,0.0,2.0,0.316667,0.683333,60.0,1,interview in Spanish,20.3,21.1,0.608470
2,300,dev,male,33,en,True,0.98,0.16,so I'm going to interview in Spanish okay good...,77,...,0.0,2.0,0.316667,0.683333,60.0,2,okay,23.9,24.3,0.690606
3,300,dev,male,33,en,True,0.98,0.16,so I'm going to interview in Spanish okay good...,77,...,0.0,2.0,0.316667,0.683333,60.0,3,good,62.1,62.7,0.951897
4,300,dev,male,33,en,True,0.98,0.16,so I'm going to interview in Spanish okay good...,77,...,0.0,2.0,0.316667,0.683333,60.0,4,Atlanta Georgia,68.8,69.8,0.987629


In [53]:
sents = sents.loc[:,["participant_id", "transcript_text", "utterance", "utterance_id"]].copy()
sents.head()

,participant_id,transcript_text,utterance,utterance_id
0,300,so I'm going to interview in Spanish okay good...,so I'm going to,0
1,300,so I'm going to interview in Spanish okay good...,interview in Spanish,1
2,300,so I'm going to interview in Spanish okay good...,okay,2
3,300,so I'm going to interview in Spanish okay good...,good,3
4,300,so I'm going to interview in Spanish okay good...,Atlanta Georgia,4


# Score for FTR

In [54]:
import json
spacy.prefer_gpu(1)
spacy.require_gpu(1)

ftr_mod = spacy.load("en_ftr_model_roberta")  # pulled from https://huggingface.co/colerobertson/en_ftr_model_roberta

def estimate_ftr(df, ftr_mod, text_col="utterance"):
    result = pd.DataFrame([doc.cats for doc in ftr_mod.pipe(df.loc[:,text_col])],index=df.index)
    return pd.concat([df,result],axis=1)

@timer_func
def ftr_score(df, ftr_mod, chunk_col, lang_col="language",text_col="utterance"):
    dx = df.copy()
    
    #split into sentences
    dfg = dx.groupby(chunk_col)
    df_sent = pd.DataFrame()
    idx = 0
    count = {}
    for chunk_col_val,dz in prog_bar(dfg,desc="Progress scoring"):
        dz_scored = estimate_ftr(dz,ftr_mod,text_col=text_col)   
        df_sent = pd.concat([df_sent,dz_scored],axis=0)
        count["idx"] = idx
        idx += 1
    df_sent["FTR_cat"] = np.where(df_sent.FTR > 0.5,1,0)
    return df_sent

In [55]:
ftr_mod("i could go").cats

{'FTR': 0.9401164054870605, 'NOT_FTR': 0.05988353490829468}

In [56]:
sent_scored = ftr_score(sents, ftr_mod, "participant_id")

Progress scoring: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 275/275 [00:50<00:00,  5.48it/s]

Function 'ftr_score' executed in 50.1547s


In [58]:
sent_scored.head()

,participant_id,transcript_text,utterance,utterance_id,FTR,NOT_FTR,FTR_cat
0,300,so I'm going to interview in Spanish okay good...,so I'm going to,0,0.940126,0.059874,1
1,300,so I'm going to interview in Spanish okay good...,interview in Spanish,1,0.009685,0.990315,0
2,300,so I'm going to interview in Spanish okay good...,okay,2,0.009948,0.990052,0
3,300,so I'm going to interview in Spanish okay good...,good,3,0.011821,0.988179,0
4,300,so I'm going to interview in Spanish okay good...,Atlanta Georgia,4,0.012608,0.987392,0


# Score for FTR type

In [59]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import ftr_classifier as ftr

ftr_df = sent_scored.loc[sent_scored.FTR_cat == 1,:].copy()
ftr_df["language"] = "english"

@timer_func
def ftr_type_score(df,chunk_col,text_col,experimental_data,clean=True,debug=False):
    dx = df.copy()
    
    #split into sentences
    dfg = dx.groupby(chunk_col,sort=False)
    df_result = pd.DataFrame()
    idx = 0
    count = {}
    for chunk_col_val,dz in prog_bar(dfg,desc="Progress scoring"):
        dz_class = ftr.classify_df(dz,text_col=text_col,experimental_data=experimental_data,debug=debug)
        if clean:
            dz_class = ftr.clean_spacy(dz_class)
        df_result = pd.concat([df_result,dz_class],axis=0)
        count["idx"] = idx
        idx += 1
    return df_result

ftr_type = ftr_type_score(ftr_df,"participant_id","utterance",False,debug=False)

Progress scoring: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 273/273 [00:45<00:00,  6.04it/s]

Function 'ftr_type_score' executed in 45.2386s


In [60]:
ftr_type.FTR_cat.value_counts()

1    2087
Name: FTR_cat, dtype: int64

In [61]:
ftr_type.shape

(2087, 38)

### Aggregate back together and merge

In [62]:
ftr_prop = sent_scored.groupby("participant_id").agg(
    FTR_proportion = ("FTR_cat", "mean")
)

In [63]:
ftr_prop.head()

,FTR_proportion
participant_id,
300,0.064935
301,0.083333
302,0.050505
303,0.106383
304,0.100000


In [64]:
ftr_type_prop = ftr_type.groupby("participant_id").agg(
      FTR_bouletic_prop = ("verb_des_int", "mean"),
      FTR_uncertain_prop = ("uncertain", "mean"),
      FTR_certain_prop = ("certain", "mean"),
      FTR_lexi_poss_prop = ("lexi_poss", "mean"),
      FTR_lexi_cert_prop = ("lexi_cert", "mean"),
      FTR_future_dom_prop = ("future_dom", "mean"),
      FTR_present_dom_prop = ("present_dom", "mean"),
      FTR_verb_cert_prop = ("verb_cert", "mean"),
      FTR_verb_poss_prop = ("verb_poss", "mean"),
      FTR_mental_poss_prop = ("mental_poss", "mean"),
  )

In [65]:
ftr_type_prop.head()

,FTR_bouletic_prop,FTR_uncertain_prop,FTR_certain_prop,FTR_lexi_poss_prop,FTR_lexi_cert_prop,FTR_future_dom_prop,FTR_present_dom_prop,FTR_verb_cert_prop,FTR_verb_poss_prop,FTR_mental_poss_prop
participant_id,,,,,,,,,,
300,0.400,0.40,0.000,0.20,0.000,0.200000,0.0,0.000000,0.2,0.000000
301,0.000,0.50,0.000,0.50,0.000,0.166667,0.0,0.166667,0.0,0.333333
302,0.600,0.20,0.000,0.20,0.000,0.000000,0.0,0.200000,0.0,0.200000
303,0.400,0.50,0.100,0.50,0.100,0.300000,0.0,0.200000,0.0,0.300000
304,0.625,0.25,0.125,0.25,0.125,0.000000,0.0,0.000000,0.0,0.000000


In [66]:
ftr_prop = ftr_prop.reset_index()
ftr_type_prop = ftr_type_prop.reset_index()

In [67]:
df_write = pd.merge(df, ftr_prop, on="participant_id", how="outer", indicator=True)
df_write._merge.value_counts()


both          275
left_only       0
right_only      0
Name: _merge, dtype: int64

In [68]:
df_write.drop("_merge",axis=1, inplace=True)

In [69]:
df_write = pd.merge(df_write, ftr_type_prop, on="participant_id", how="outer", indicator=True)
df_write._merge.value_counts()

both          273
left_only       2
right_only      0
Name: _merge, dtype: int64

In [72]:
df_write.head()

,participant_id,split,gender,age,language,is_english,en_token_ratio,es_token_ratio,transcript_text,n_utterances,...,FTR_uncertain_prop,FTR_certain_prop,FTR_lexi_poss_prop,FTR_lexi_cert_prop,FTR_future_dom_prop,FTR_present_dom_prop,FTR_verb_cert_prop,FTR_verb_poss_prop,FTR_mental_poss_prop,_merge
0,300,dev,male,33,en,True,0.98,0.16,so I'm going to interview in Spanish okay good...,77,...,0.40,0.000,0.20,0.000,0.200000,0.0,0.000000,0.2,0.000000,both
1,301,dev,male,39,en,True,0.93,0.11,yeah there's also on Craigslist so that's why ...,72,...,0.50,0.000,0.50,0.000,0.166667,0.0,0.166667,0.0,0.333333,both
2,302,train,male,25,en,True,0.99,0.10,just move around a little bit when you're fini...,99,...,0.20,0.000,0.20,0.000,0.000000,0.0,0.200000,0.0,0.200000,both
3,303,train,female,41,en,True,0.96,0.09,wow okay when you're finished when she's done ...,94,...,0.50,0.100,0.50,0.100,0.300000,0.0,0.200000,0.0,0.300000,both
4,304,train,female,22,en,True,0.96,0.11,so we'll just move around a little bit tonight...,80,...,0.25,0.125,0.25,0.125,0.000000,0.0,0.000000,0.0,0.000000,both


In [71]:
# Write results
df_write.to_csv("../data/edaic_ALL_participant_features_FTR.csv", index=False)
ftr_type.to_csv("../data/utterance_scored_ftr_type_features_for_future_referring_sentences.csv", index=False)
ftr_type_prop.to_csv("../data/utterance_scored_ftr_probability.csv", index=False)
